In [1]:
import tensorflow as tf
import os
import numpy as np
# from osgeo import gdal, osr
# import cv2
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torch
from transformers import Mask2FormerForUniversalSegmentation, AutoConfig, Mask2FormerConfig
import torch.nn as nn

2026-01-31 12:26:29.012884: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/glade/derecho/scratch/lizhili/CFAT/lib/python3.10/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.26.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/glade/derecho/scratch/lizhili/CFAT/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
import torch
from torch import nn
import torch.nn.functional as F


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x = self.conv(x)
        return x


class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(Up, self).__init__()
        self.up_scale = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)

    def forward(self, x1, x2):
        x2 = self.up_scale(x2)

        diffY = x1.size()[2] - x2.size()[2]
        diffX = x1.size()[3] - x2.size()[3]

        x2 = F.pad(x2, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return x


class DownLayer(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(DownLayer, self).__init__()
        self.pool = nn.MaxPool2d(2, stride=2, padding=0)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x):
        x = self.conv(self.pool(x))
        return x


class UpLayer(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(UpLayer, self).__init__()
        self.up = Up(in_ch, out_ch)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x1, x2):
        a = self.up(x1, x2)
        x = self.conv(a)
        return x


class UNet(nn.Module):
    def __init__(self, input_dims = 4, output_dims=2):
        super(UNet, self).__init__()
        self.conv1 = DoubleConv(input_dims, 64)
        self.down1 = DownLayer(64, 128)
        self.down2 = DownLayer(128, 256)
        self.down3 = DownLayer(256, 512)
        self.down4 = DownLayer(512, 1024)
        self.up1 = UpLayer(1024, 512)
        self.up2 = UpLayer(512, 256)
        self.up3 = UpLayer(256, 128)
        self.up4 = UpLayer(128, 64)
        self.last_conv = nn.Conv2d(64, output_dims, 1)

    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x1_up = self.up1(x4, x5)
        x2_up = self.up2(x3, x1_up)
        x3_up = self.up3(x2, x2_up)
        x4_up = self.up4(x1, x3_up)
        output = self.last_conv(x4_up)
        return output

In [18]:
import torch

def random_crop_image_label(
    image,
    label,
    crop_size
):
    """
    Random aligned crop for image–label pairs with strict size checking.

    Args:
        image: Tensor [B, C, H, W]
        label: Tensor [B, H, W] or [B, 1, H, W]
        crop_size: int

    Returns:
        image_crop: [B, C, crop_size, crop_size]
        label_crop: [B, crop_size, crop_size]
    """
    # ---- Shape checks ----
    assert image.dim() == 4, f"image must be [B, C, H, W], got {image.shape}"
    assert label.dim() in (3, 4), f"label must be [B, H, W] or [B, 1, H, W], got {label.shape}"

    _, _, H_img, W_img = image.shape

    if label.dim() == 3:
        _, H_lbl, W_lbl = label.shape
    else:
        _, _, H_lbl, W_lbl = label.shape

    # ---- Enforce same spatial size ----
    assert H_img == H_lbl and W_img == W_lbl, (
        f"Image and label spatial sizes must match, "
        f"got image ({H_img}, {W_img}) and label ({H_lbl}, {W_lbl})"
    )

    assert H_img >= crop_size and W_img >= crop_size, (
        f"Crop size {crop_size} exceeds image size ({H_img}, {W_img})"
    )

    # ---- Random crop ----
    top = torch.randint(0, H_img - crop_size + 1, (1,)).item()
    left = torch.randint(0, W_img - crop_size + 1, (1,)).item()

    image_crop = image[:, :, top:top + crop_size, left:left + crop_size]

    if label.dim() == 3:
        label = label.unsqueeze(1)

    label_crop = label[:, :, top:top + crop_size, left:left + crop_size]
    label_crop = label_crop.squeeze(1).long()

    return image_crop, label_crop

In [19]:
def input_pipeline_downstream_sr(filename, batch_size, skip, take, is_shuffle=True, is_train=True, is_repeat=True):
        feature_description = {
            'lres': tf.io.FixedLenFeature([7*lres_size*lres_size], dtype=tf.int64),
            'hres': tf.io.FixedLenFeature([7*hres_size*hres_size], dtype=tf.int64),
            'label': tf.io.FixedLenFeature([label_size*label_size], dtype=tf.int64)
        }

        @tf.function
        def _parse_function(example_proto):
            feature_dict = tf.io.parse_single_example(example_proto, feature_description)

            lres = feature_dict['lres']
            lres = tf.reshape(lres, [7, lres_size, lres_size])
            lres = tf.cast(lres, tf.float32)*0.0001

            hres = feature_dict['hres']
            hres = tf.reshape(hres, [7, hres_size, hres_size])
            hres = tf.cast(hres, tf.float32)*0.0000275-0.2

            label = feature_dict['label']
            label = tf.reshape(label, [label_size, label_size, 1])
            return lres, hres, label[..., 0]

        @tf.function
        def _augment_function(lres_img, hres_img, label):
            # Transpose to [H, W, C]
            lres_img = tf.transpose(lres_img, [1, 2, 0])
            hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]
            if tf.rank(label) == 2:
                label = tf.expand_dims(label, axis=-1)
        
            # Randomly choose 0, 90, 180, or 270 degrees
            k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
            lres_img = tf.image.rot90(lres_img, k=k)
            hres_img = tf.image.rot90(hres_img, k=k)
            label = tf.image.rot90(label, k=k)
    
            # ---- Random horizontal flip ----
            do_flip_lr = tf.random.uniform([]) > 0.5
            lres_img = tf.cond(do_flip_lr,
                               lambda: tf.image.flip_left_right(lres_img),
                               lambda: lres_img)
            hres_img = tf.cond(do_flip_lr,
                               lambda: tf.image.flip_left_right(hres_img),
                               lambda: hres_img)
            label = tf.cond(do_flip_lr,
                            lambda: tf.image.flip_left_right(label),
                            lambda: label)
        
            # ---- Random vertical flip ----
            do_flip_ud = tf.random.uniform([]) > 0.5
            lres_img = tf.cond(do_flip_ud,
                               lambda: tf.image.flip_up_down(lres_img),
                               lambda: lres_img)
            hres_img = tf.cond(do_flip_ud,
                               lambda: tf.image.flip_up_down(hres_img),
                               lambda: hres_img)
            label = tf.cond(do_flip_ud,
                            lambda: tf.image.flip_up_down(label),
                            lambda: label)
        
            # Transpose back to [C, H, W]
            lres_img = tf.transpose(lres_img, [2, 0, 1])
            hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]
            label = tf.squeeze(label, axis=-1)
        
            return lres_img, hres_img, label

        dataset = tf.data.TFRecordDataset(filename)
        dataset = dataset.skip(skip)
        if take:
            dataset = dataset.take(take)
        if is_repeat:
            dataset = dataset.repeat()
        dataset = dataset.map(_parse_function)
        if is_train:
            dataset = dataset.map(_augment_function)
        if is_shuffle:
            dataset = dataset.shuffle(buffer_size=100)
        batch = dataset.batch(batch_size=batch_size)

        return batch

# filenames = '/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_CDL.tfrecords'
# ds = input_pipeline_downstream_sr(filenames, 10, 0, 200, is_shuffle=True, is_train=True, is_repeat=True)


# for lres_batch, hres_batch, label in ds:
#     print(lres_batch.shape)
#     print(hres_batch.shape)

#     for i in range(lres_batch.shape[0]):
#         fig, axes = plt.subplots(1, 3, figsize=(8, 4))

#         lres_img = np.transpose(lres_batch[i].numpy(), (1, 2, 0))
#         hres_img = np.transpose(hres_batch[i].numpy(), (1, 2, 0))
#         label_img = label[i].numpy()

#         axes[0].imshow(lres_img[:, :, [0,3,2]]*3)
#         axes[0].set_title('Low-Resolution')
#         axes[0].axis('off')

#         axes[1].imshow(hres_img[:, :, 3:0:-1]*3)
#         axes[1].set_title('High-Resolution')
#         axes[1].axis('off')

#         axes[2].imshow(label_img)
#         axes[2].set_title('label')
#         axes[2].axis('off')

#         plt.show()

#     break

In [20]:
def unet_run(lres_size ,
                    hres_size ,
                    hres_size_4x ,
                    label_size ,
                    num_sample,
                    num_training ,
                    class_num,
                    start_class,
                    finetune_tfrecords,
                    unet_s2_save_path,
                    unet_naip_save_path
             ):

    num_test = num_sample-num_training

    def input_pipeline_downstream_sr(filename, batch_size, skip, take, is_shuffle=True, is_train=True, is_repeat=True):
        feature_description = {
            'lres': tf.io.FixedLenFeature([7*lres_size*lres_size], dtype=tf.int64),
            'hres': tf.io.FixedLenFeature([7*hres_size*hres_size], dtype=tf.int64),
            'label': tf.io.FixedLenFeature([label_size*label_size], dtype=tf.int64)
        }

        @tf.function
        def _parse_function(example_proto):
            feature_dict = tf.io.parse_single_example(example_proto, feature_description)

            lres = feature_dict['lres']
            lres = tf.reshape(lres, [7, lres_size, lres_size])
            lres = tf.cast(lres, tf.float32)*0.0001

            hres = feature_dict['hres']
            hres = tf.reshape(hres, [7, hres_size, hres_size])
            hres = tf.cast(hres, tf.float32)*0.0000275-0.2

            label = feature_dict['label']
            label = tf.reshape(label, [label_size, label_size, 1])
            return lres, hres, label[..., 0]

        @tf.function
        def _augment_function(lres_img, hres_img, label):
            # Transpose to [H, W, C]
            lres_img = tf.transpose(lres_img, [1, 2, 0])
            hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]
            if tf.rank(label) == 2:
                label = tf.expand_dims(label, axis=-1)
        
            # Randomly choose 0, 90, 180, or 270 degrees
            k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
            lres_img = tf.image.rot90(lres_img, k=k)
            hres_img = tf.image.rot90(hres_img, k=k)
            label = tf.image.rot90(label, k=k)
    
            # ---- Random horizontal flip ----
            do_flip_lr = tf.random.uniform([]) > 0.5
            lres_img = tf.cond(do_flip_lr,
                               lambda: tf.image.flip_left_right(lres_img),
                               lambda: lres_img)
            hres_img = tf.cond(do_flip_lr,
                               lambda: tf.image.flip_left_right(hres_img),
                               lambda: hres_img)
            label = tf.cond(do_flip_lr,
                            lambda: tf.image.flip_left_right(label),
                            lambda: label)
        
            # ---- Random vertical flip ----
            do_flip_ud = tf.random.uniform([]) > 0.5
            lres_img = tf.cond(do_flip_ud,
                               lambda: tf.image.flip_up_down(lres_img),
                               lambda: lres_img)
            hres_img = tf.cond(do_flip_ud,
                               lambda: tf.image.flip_up_down(hres_img),
                               lambda: hres_img)
            label = tf.cond(do_flip_ud,
                            lambda: tf.image.flip_up_down(label),
                            lambda: label)
        
            # Transpose back to [C, H, W]
            lres_img = tf.transpose(lres_img, [2, 0, 1])
            hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]
            label = tf.squeeze(label, axis=-1)
        
            return lres_img, hres_img, label

        dataset = tf.data.TFRecordDataset(filename)
        dataset = dataset.skip(skip)
        if take:
            dataset = dataset.take(take)
        if is_repeat:
            dataset = dataset.repeat()
        dataset = dataset.map(_parse_function)
        if is_train:
            dataset = dataset.map(_augment_function)
        if is_shuffle:
            dataset = dataset.shuffle(buffer_size=100)
        batch = dataset.batch(batch_size=batch_size)

        return batch

    # --------------------------------------------
    print('Begin S2 Unet Training')
    device = 'cuda'
    model_s2 = UNet(input_dims=7, output_dims=class_num).to(device)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model_s2.parameters(), lr=1e-4)

    # Training loop
    def train_step(images, labels):
        model_s2.train()

        images = torch.from_numpy(images.numpy().astype('float32'))
        images = F.interpolate(images, size=(64, 64), mode='bilinear')

        labels = torch.from_numpy(labels.numpy())
        labels = labels.float().unsqueeze(1)
        labels = F.interpolate(labels, size=(64, 64), mode='nearest')
        labels = labels.squeeze(1).to(torch.int64)

        images = images.to(device)  # [B, 4, 256, 256]
        labels = labels.to(device)  # [B, 1000, 1000] as class indices

        optimizer.zero_grad()
        logits = model_s2(images)  # [B, 13, 512, 512]
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        return loss.item()

    @torch.no_grad()
    def val_step(images, labels):
        model_s2.eval()
    
        images = torch.from_numpy(images.numpy().astype('float32'))
        images = F.interpolate(images, size=(64, 64), mode='bilinear')
        
        labels = torch.from_numpy(labels.numpy())
        labels = labels.float().unsqueeze(1)
        labels = F.interpolate(labels, size=(64, 64), mode='nearest')
        labels = labels.squeeze(1).to(torch.int64)
    
        images = images.to(device)
        labels = labels.to(device)
    
        logits = model_s2(images)
        loss = loss_fn(logits, labels)
    
        return loss.item()

    best_val_loss = float('inf')

    def train_val_epoch(train_loader, val_loader, epoch):
        nonlocal best_val_loss
    
        # ---- Training ----
        total_train_loss = 0.0
        for lr, _, label in train_loader:
            loss = train_step(lr, label)
            total_train_loss += loss
    
        avg_train_loss = total_train_loss / num_train
    
        avg_val_loss = None  # safeguard
    
        # ---- Validation (every 5 epochs) ----
        if (epoch + 1) % 5 == 0:
            total_val_loss = 0.0
            for lr, _, label in val_loader:
                loss = val_step(lr, label)
                total_val_loss += loss
    
            avg_val_loss = total_val_loss / num_val
    
            # ---- Checkpoint best model ----
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                torch.save(
                    model_s2.state_dict(),
                    unet_s2_save_path.replace(".pth", "_best.pth")
                )
                print("✓ Saved new best model")
    
        # ---- Logging ----
        if (epoch + 1) % 10 == 0:
            log_msg = f"Epoch {epoch+1:03d} | Train Loss: {avg_train_loss:.4f}"
            if avg_val_loss is not None:
                log_msg += f" | Val Loss: {avg_val_loss:.4f}"
            print(log_msg)

    num_val = int(0.1 * num_training)   # 10% for validation
    num_train = num_training - num_val

    train_loader = input_pipeline_downstream_sr(finetune_tfrecords, 8, 0, num_train, is_repeat=False)
    val_loader = input_pipeline_downstream_sr(finetune_tfrecords, 2, num_train, num_val, is_shuffle=False, is_train=False,  is_repeat=False)
    # train_loader = input_pipeline_downstream_sr(finetune_tfrecords, 20, 0, num_training, is_repeat=False)

    for i in range(100):
        train_val_epoch(train_loader, val_loader, i)

    # --------------------------------------------
    print('Begin NAIP Unet Training')
    device = 'cuda'
    model_naip = UNet(input_dims=7, output_dims=class_num).to(device)
    # model_naip.load_state_dict(torch.load(unet_save_path, weights_only=True))

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model_naip.parameters(), lr=1e-4)

    # Training loop
    def train_step(images, labels):
        model_naip.train()

        images = torch.from_numpy(images.numpy().astype('float32'))
        images = F.interpolate(images, size=(hres_size_4x, hres_size_4x), mode='bilinear')

        labels = torch.from_numpy(labels.numpy())
        labels = labels.float().unsqueeze(1)
        labels = F.interpolate(labels, size=(hres_size_4x, hres_size_4x), mode='nearest')
        labels = labels.squeeze(1).to(torch.int64)

        images, labels = random_crop_image_label(images, labels, crop_size = 256)

        images = images.to(device)  # [B, 4, 256, 256]
        labels = labels.to(device)  # [B, 1000, 1000] as class indices

        optimizer.zero_grad()
        logits = model_naip(images)  # [B, 13, 512, 512]
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        return loss.item()

    @torch.no_grad()
    def val_step(images, labels):
        model_naip.eval()
    
        images = torch.from_numpy(images.numpy().astype('float32'))
        images = F.interpolate(images, size=(hres_size_4x, hres_size_4x), mode='bilinear')

        labels = torch.from_numpy(labels.numpy())
        labels = labels.float().unsqueeze(1)
        labels = F.interpolate(labels, size=(hres_size_4x, hres_size_4x), mode='nearest')
        labels = labels.squeeze(1).to(torch.int64)
    
        images = images.to(device)
        labels = labels.to(device)
    
        logits = model_naip(images)
        loss = loss_fn(logits, labels)
    
        return loss.item()

    best_val_loss = float('inf')

    def train_val_epoch(train_loader, val_loader, epoch):
        nonlocal best_val_loss
    
        # ---- Training ----
        total_train_loss = 0.0
        for _, hr, label in train_loader:
            loss = train_step(hr, label)
            total_train_loss += loss
    
        avg_train_loss = total_train_loss / num_train
    
        avg_val_loss = None  # safeguard
    
        # ---- Validation (every 5 epochs) ----
        if (epoch + 1) % 5 == 0:
            total_val_loss = 0.0
            for _, hr, label in val_loader:
                loss = val_step(hr, label)
                total_val_loss += loss
    
            avg_val_loss = total_val_loss / num_val
    
            # ---- Checkpoint best model ----
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                torch.save(
                    model_naip.state_dict(),
                    unet_naip_save_path.replace(".pth", "_best.pth")
                )
                print("✓ Saved new best model")
    
        # ---- Logging ----
        if (epoch + 1) % 10 == 0:
            log_msg = f"Epoch {epoch+1:03d} | Train Loss: {avg_train_loss:.4f}"
            if avg_val_loss is not None:
                log_msg += f" | Val Loss: {avg_val_loss:.4f}"
            print(log_msg)

    num_val = int(0.1 * num_training)   # 10% for validation
    num_train = num_training - num_val

    train_loader = input_pipeline_downstream_sr(finetune_tfrecords, 8, 0, num_train, is_repeat=False)
    val_loader = input_pipeline_downstream_sr(finetune_tfrecords, 2, num_train, num_val, is_shuffle=False, is_train=False,  is_repeat=False)
    # train_loader = input_pipeline_downstream_sr(finetune_tfrecords, 20, 0, num_training, is_repeat=False)

    for i in range(100):
        train_val_epoch(train_loader, val_loader, i)



In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 1065
num_training = 852
class_num=2
start_class=0
num_test = num_sample-num_training
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_River.tfrecords']
unet_s2_save_path = '/glade/derecho/scratch/lizhili/m2l8/M2L8_River_M2_Unet_run1.pth'
unet_naip_save_path = '/glade/derecho/scratch/lizhili/m2l8/M2L8_River_L8_Unet_run1.pth'

unet_run(lres_size ,
         hres_size ,
         hres_size_4x ,
         label_size ,
         num_sample,
         num_training ,
         class_num,
         start_class,
         finetune_tfrecords,
         unet_s2_save_path,
         unet_naip_save_path
        )

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 1687
num_training = 1350
class_num=11
num_test = num_sample-num_training
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_Urban.tfrecords']
unet_s2_save_path = '/glade/derecho/scratch/lizhili/m2l8/M2L8_Urban_M2_Unet_run1.pth'
unet_naip_save_path = '/glade/derecho/scratch/lizhili/m2l8/M2L8_Urban_L8_Unet_run1.pth'
start_class = 0

unet_run(lres_size ,
         hres_size ,
         hres_size_4x ,
         label_size ,
         num_sample,
         num_training ,
         class_num,
         start_class,
         finetune_tfrecords,
         unet_s2_save_path,
         unet_naip_save_path
        )

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 762
num_training = 610
class_num=5
num_test = num_sample-num_training
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_CDL.tfrecords']
unet_s2_save_path = '/glade/derecho/scratch/lizhili/m2l8/M2L8_CDL_M2_Unet_run1.pth'
unet_naip_save_path = '/glade/derecho/scratch/lizhili/m2l8/M2L8_CDL_L8_Unet_run1.pth'
start_class = 0

unet_run(lres_size ,
         hres_size ,
         hres_size_4x ,
         label_size ,
         num_sample,
         num_training ,
         class_num,
         start_class,
         finetune_tfrecords,
         unet_s2_save_path,
         unet_naip_save_path
        )

# Regression

In [4]:
import torch

def random_crop_image_label(
    image,
    label,
    crop_size
):
    """
    Random aligned crop for image–label pairs with strict size checking.

    Args:
        image: Tensor [B, C, H, W]
        label: Tensor [B, H, W] or [B, 1, H, W]
        crop_size: int

    Returns:
        image_crop: [B, C, crop_size, crop_size]
        label_crop: [B, crop_size, crop_size]
    """
    # ---- Shape checks ----
    assert image.dim() == 4, f"image must be [B, C, H, W], got {image.shape}"
    assert label.dim() in (3, 4), f"label must be [B, H, W] or [B, 1, H, W], got {label.shape}"

    _, _, H_img, W_img = image.shape

    if label.dim() == 3:
        _, H_lbl, W_lbl = label.shape
    else:
        _, _, H_lbl, W_lbl = label.shape

    # ---- Enforce same spatial size ----
    assert H_img == H_lbl and W_img == W_lbl, (
        f"Image and label spatial sizes must match, "
        f"got image ({H_img}, {W_img}) and label ({H_lbl}, {W_lbl})"
    )

    assert H_img >= crop_size and W_img >= crop_size, (
        f"Crop size {crop_size} exceeds image size ({H_img}, {W_img})"
    )

    # ---- Random crop ----
    top = torch.randint(0, H_img - crop_size + 1, (1,)).item()
    left = torch.randint(0, W_img - crop_size + 1, (1,)).item()

    image_crop = image[:, :, top:top + crop_size, left:left + crop_size]

    if label.dim() == 3:
        label = label.unsqueeze(1)

    label_crop = label[:, :, top:top + crop_size, left:left + crop_size]

    return image_crop, label_crop

In [5]:
def unet_run_regression(lres_size ,
                    hres_size ,
                    hres_size_4x ,
                    label_size ,
                    num_sample,
                    num_training ,
                    class_num,
                    finetune_tfrecords,
                    unet_s2_save_path,
                    unet_naip_save_path,
                    value_multiplier
             ):

    num_test = num_sample-num_training

    def input_pipeline_downstream_sr(filename, batch_size, skip, take, is_shuffle=True, is_train=True, is_repeat=True):
        feature_description = {
            'lres': tf.io.FixedLenFeature([7*lres_size*lres_size], dtype=tf.int64),
            'hres': tf.io.FixedLenFeature([7*hres_size*hres_size], dtype=tf.int64),
            'label': tf.io.FixedLenFeature([label_size*label_size], dtype=tf.int64)
        }

        @tf.function
        def _parse_function(example_proto):
            feature_dict = tf.io.parse_single_example(example_proto, feature_description)

            lres = feature_dict['lres']
            lres = tf.reshape(lres, [7, lres_size, lres_size])
            lres = tf.cast(lres, tf.float32)*0.0001

            hres = feature_dict['hres']
            hres = tf.reshape(hres, [7, hres_size, hres_size])
            hres = tf.cast(hres, tf.float32)*0.0000275-0.2

            label = feature_dict['label']
            label = tf.reshape(label, [label_size, label_size, 1])
            label = tf.cast(label, tf.float32)*value_multiplier
            return lres, hres, label[..., 0]

        @tf.function
        def _augment_function(lres_img, hres_img, label):
            # Transpose to [H, W, C]
            lres_img = tf.transpose(lres_img, [1, 2, 0])
            hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]
            if tf.rank(label) == 2:
                label = tf.expand_dims(label, axis=-1)
        
            # Randomly choose 0, 90, 180, or 270 degrees
            k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
            lres_img = tf.image.rot90(lres_img, k=k)
            hres_img = tf.image.rot90(hres_img, k=k)
            label = tf.image.rot90(label, k=k)
    
            # ---- Random horizontal flip ----
            do_flip_lr = tf.random.uniform([]) > 0.5
            lres_img = tf.cond(do_flip_lr,
                               lambda: tf.image.flip_left_right(lres_img),
                               lambda: lres_img)
            hres_img = tf.cond(do_flip_lr,
                               lambda: tf.image.flip_left_right(hres_img),
                               lambda: hres_img)
            label = tf.cond(do_flip_lr,
                            lambda: tf.image.flip_left_right(label),
                            lambda: label)
        
            # ---- Random vertical flip ----
            do_flip_ud = tf.random.uniform([]) > 0.5
            lres_img = tf.cond(do_flip_ud,
                               lambda: tf.image.flip_up_down(lres_img),
                               lambda: lres_img)
            hres_img = tf.cond(do_flip_ud,
                               lambda: tf.image.flip_up_down(hres_img),
                               lambda: hres_img)
            label = tf.cond(do_flip_ud,
                            lambda: tf.image.flip_up_down(label),
                            lambda: label)
        
            # Transpose back to [C, H, W]
            lres_img = tf.transpose(lres_img, [2, 0, 1])
            hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]
            label = tf.squeeze(label, axis=-1)
        
            return lres_img, hres_img, label

        dataset = tf.data.TFRecordDataset(filename)
        dataset = dataset.skip(skip)
        if take:
            dataset = dataset.take(take)
        if is_repeat:
            dataset = dataset.repeat()
        dataset = dataset.map(_parse_function)
        if is_train:
            dataset = dataset.map(_augment_function)
        if is_shuffle:
            dataset = dataset.shuffle(buffer_size=100)
        batch = dataset.batch(batch_size=batch_size)

        return batch

    # --------------------------------------------
    print('Begin S2 Unet Training')
    device = 'cuda'
    model_s2 = UNet(input_dims=7, output_dims=class_num).to(device)

    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model_s2.parameters(), lr=1e-4)

    # Training loop
    def train_step(images, labels):
        model_s2.train()

        images = torch.from_numpy(images.numpy().astype('float32'))
        images = F.interpolate(images, size=(64, 64), mode='bilinear')

        labels = torch.from_numpy(labels.numpy())
        labels = labels.float().unsqueeze(1)
        labels = F.interpolate(labels, size=(64, 64), mode='nearest')

        images = images.to(device)  # [B, 4, 256, 256]
        labels = labels.to(device)  # [B, 1000, 1000] as class indices

        optimizer.zero_grad()
        logits = model_s2(images)  # [B, 13, 512, 512]
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        return loss.item()

    @torch.no_grad()
    def val_step(images, labels):
        model_s2.eval()
    
        images = torch.from_numpy(images.numpy().astype('float32'))
        images = F.interpolate(images, size=(64, 64), mode='bilinear')
        
        labels = torch.from_numpy(labels.numpy())
        labels = labels.float().unsqueeze(1)
        labels = F.interpolate(labels, size=(64, 64), mode='nearest')
    
        images = images.to(device)
        labels = labels.to(device)
    
        logits = model_s2(images)
        loss = loss_fn(logits, labels)
    
        return loss.item()

    best_val_loss = float('inf')

    def train_val_epoch(train_loader, val_loader, epoch):
        nonlocal best_val_loss
    
        # ---- Training ----
        total_train_loss = 0.0
        for lr, _, label in train_loader:
            loss = train_step(lr, label)
            total_train_loss += loss
    
        avg_train_loss = total_train_loss / num_train
    
        avg_val_loss = None  # safeguard
    
        # ---- Validation (every 5 epochs) ----
        if (epoch + 1) % 5 == 0:
            total_val_loss = 0.0
            for lr, _, label in val_loader:
                loss = val_step(lr, label)
                total_val_loss += loss
    
            avg_val_loss = total_val_loss / num_val
    
            # ---- Checkpoint best model ----
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                torch.save(
                    model_s2.state_dict(),
                    unet_s2_save_path.replace(".pth", "_best.pth")
                )
                print("✓ Saved new best model")
    
        # ---- Logging ----
        if (epoch + 1) % 10 == 0:
            log_msg = f"Epoch {epoch+1:03d} | Train Loss: {avg_train_loss:.4f}"
            if avg_val_loss is not None:
                log_msg += f" | Val Loss: {avg_val_loss:.4f}"
            print(log_msg)

    num_val = int(0.1 * num_training)   # 10% for validation
    num_train = num_training - num_val

    train_loader = input_pipeline_downstream_sr(finetune_tfrecords, 8, 0, num_train, is_repeat=False)
    val_loader = input_pipeline_downstream_sr(finetune_tfrecords, 2, num_train, num_val, is_shuffle=False, is_train=False,  is_repeat=False)
    # train_loader = input_pipeline_downstream_sr(finetune_tfrecords, 20, 0, num_training, is_repeat=False)

    for i in range(100):
        train_val_epoch(train_loader, val_loader, i)

    # --------------------------------------------
    print('Begin NAIP Unet Training')
    device = 'cuda'
    model_naip = UNet(input_dims=7, output_dims=class_num).to(device)
    # model_naip.load_state_dict(torch.load(unet_save_path, weights_only=True))

    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model_naip.parameters(), lr=1e-4)

    # Training loop
    def train_step(images, labels):
        model_naip.train()

        images = torch.from_numpy(images.numpy().astype('float32'))
        images = F.interpolate(images, size=(hres_size_4x, hres_size_4x), mode='bilinear')

        labels = torch.from_numpy(labels.numpy())
        labels = labels.float().unsqueeze(1)
        labels = F.interpolate(labels, size=(hres_size_4x, hres_size_4x), mode='nearest')

        images, labels = random_crop_image_label(images, labels, crop_size = 256)

        images = images.to(device)  # [B, 4, 256, 256]
        labels = labels.to(device)  # [B, 1000, 1000] as class indices

        optimizer.zero_grad()
        logits = model_naip(images)  # [B, 13, 512, 512]
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        return loss.item()

    @torch.no_grad()
    def val_step(images, labels):
        model_naip.eval()
    
        images = torch.from_numpy(images.numpy().astype('float32'))
        images = F.interpolate(images, size=(hres_size_4x, hres_size_4x), mode='bilinear')

        labels = torch.from_numpy(labels.numpy())
        labels = labels.float().unsqueeze(1)
        labels = F.interpolate(labels, size=(hres_size_4x, hres_size_4x), mode='nearest')
    
        images = images.to(device)
        labels = labels.to(device)
    
        logits = model_naip(images)
        loss = loss_fn(logits, labels)
    
        return loss.item()

    best_val_loss = float('inf')

    def train_val_epoch(train_loader, val_loader, epoch):
        nonlocal best_val_loss
    
        # ---- Training ----
        total_train_loss = 0.0
        for _, hr, label in train_loader:
            loss = train_step(hr, label)
            total_train_loss += loss
    
        avg_train_loss = total_train_loss / num_train
    
        avg_val_loss = None  # safeguard
    
        # ---- Validation (every 5 epochs) ----
        if (epoch + 1) % 5 == 0:
            total_val_loss = 0.0
            for _, hr, label in val_loader:
                loss = val_step(hr, label)
                total_val_loss += loss
    
            avg_val_loss = total_val_loss / num_val
    
            # ---- Checkpoint best model ----
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                torch.save(
                    model_naip.state_dict(),
                    unet_naip_save_path.replace(".pth", "_best.pth")
                )
                print("✓ Saved new best model")
    
        # ---- Logging ----
        if (epoch + 1) % 10 == 0:
            log_msg = f"Epoch {epoch+1:03d} | Train Loss: {avg_train_loss:.4f}"
            if avg_val_loss is not None:
                log_msg += f" | Val Loss: {avg_val_loss:.4f}"
            print(log_msg)

    num_val = int(0.1 * num_training)   # 10% for validation
    num_train = num_training - num_val

    train_loader = input_pipeline_downstream_sr(finetune_tfrecords, 8, 0, num_train, is_repeat=False)
    val_loader = input_pipeline_downstream_sr(finetune_tfrecords, 2, num_train, num_val, is_shuffle=False, is_train=False,  is_repeat=False)
    # train_loader = input_pipeline_downstream_sr(finetune_tfrecords, 20, 0, num_training, is_repeat=False)

    for i in range(100):
        train_val_epoch(train_loader, val_loader, i)



In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
class_num = 1
num_sample = 755
num_training = 604
num_test = num_sample-num_training
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_GPP.tfrecords']
unet_s2_save_path = '/glade/derecho/scratch/lizhili/m2l8/M2L8_GPP_M2_Unet_run1.pth'
unet_naip_save_path = '/glade/derecho/scratch/lizhili/m2l8/M2L8_GPP_L8_Unet_run1.pth'
value_multiplier = 0.0001

unet_run_regression(lres_size ,
         hres_size ,
         hres_size_4x ,
         label_size ,
         num_sample,
         num_training ,
         class_num,
         finetune_tfrecords,
         unet_s2_save_path,
         unet_naip_save_path,
         value_multiplier
        )

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
class_num = 1
num_sample = 1408
num_training = 1126
num_test = num_sample-num_training
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_CHM.tfrecords']
unet_s2_save_path = '/glade/derecho/scratch/lizhili/m2l8/M2L8_CHM_M2_Unet_run1.pth'
unet_naip_save_path = '/glade/derecho/scratch/lizhili/m2l8/M2L8_CHM_L8_Unet_run1.pth'
value_multiplier = 0.1

unet_run_regression(lres_size ,
         hres_size ,
         hres_size_4x ,
         label_size ,
         num_sample,
         num_training ,
         class_num,
         finetune_tfrecords,
         unet_s2_save_path,
         unet_naip_save_path,
         value_multiplier
        )